In [ ]:
from huggingface_hub import login
login("")

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

In [5]:
def load_model_and_tokenizer(model_name):

    print("Loading tokenizer...")

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print("Loading model...")

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto"
    )

    model.eval()

    return model, tokenizer

In [6]:
!pip install -U bitsandbytes>=0.46.1

In [7]:
def infer(
    MODEL,
    TOKENIZER,
    prompt: str,
    debug: bool = False
):
    system_prompt = """
You are an intelligent Robot Task Planning Agent.
"""

    messages = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    # --------------------------------------------------------
    # Build prompt
    # --------------------------------------------------------

    try:
        formatted_prompt = TOKENIZER.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False
        )
    except TypeError:
        formatted_prompt = TOKENIZER.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

    # --------------------------------------------------------
    # Tokenize
    # --------------------------------------------------------

    inputs = TOKENIZER(
        formatted_prompt,
        return_tensors="pt"
    )

    inputs = {k: v.to(MODEL.device) for k, v in inputs.items()}

    # --------------------------------------------------------
    # Debug
    # --------------------------------------------------------

    if debug:

        print("=" * 80)
        print("MODEL")
        print("=" * 80)
        print(MODEL.config._name_or_path)

        print("\nTokenizer:", TOKENIZER.__class__.__name__)

        print("\nInput IDs Shape:")
        print(inputs["input_ids"].shape)

        print("\nPrompt Preview")
        print("=" * 80)
        print(formatted_prompt[:2000])
        print("=" * 80)

    # --------------------------------------------------------
    # Generate
    # --------------------------------------------------------

    with torch.no_grad():

        outputs = MODEL.generate(

            **inputs,

            max_new_tokens=3072,

            do_sample=False,

            repetition_penalty=1.05,

            use_cache=True,

            pad_token_id=TOKENIZER.eos_token_id,

            eos_token_id=TOKENIZER.eos_token_id,
        )

    # --------------------------------------------------------
    # Decode only generated tokens
    # --------------------------------------------------------

    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

    response = TOKENIZER.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    if debug:

        print("\nGenerated Tokens:", len(generated_tokens))

        print("\nResponse")
        print("=" * 80)
        print(response)
        print("=" * 80)

    return response

In [8]:
! pip install pyngrok

In [ ]:
import traceback
import nest_asyncio
import uvicorn
from pyngrok import ngrok

from pydantic import BaseModel
class PlanRequest(BaseModel):
    model_name: str
    prompt: str


MODEL = None
TOKENIZER = None
CURRENT_MODEL = None

from fastapi import FastAPI
app = FastAPI()

@app.post("/plan")
def get_plan(request: PlanRequest):

    global MODEL
    global TOKENIZER
    global CURRENT_MODEL

    if CURRENT_MODEL != request.model_name:

        print(f"XXXXXXXX LOADING MODEL: {request.model_name}")

        MODEL, TOKENIZER = load_model_and_tokenizer(
            request.model_name
        )

        CURRENT_MODEL = request.model_name

    print(next(MODEL.parameters()).device)

    print(f"GPU memory allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

    print(f"GPU memory reserved: {torch.cuda.memory_reserved()/1024**3:.2f} GB")

    # print(f"======== INFERENCE: {request.prompt}")

    try:

        from datetime import datetime

        start = datetime.now()

        print(f"INFER BEGIN AT: {start}")

        response = infer(
            MODEL,
            TOKENIZER,
            request.prompt,
        )

        print(f"INFER END AT: {datetime.now()}")

        print(f"INFER TIME TAKEN: {datetime.now() - start}")

        print(f"Peak GPU memory: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")

        print("======== INFERENCE: Sending response")

        # print("======== RESPONSE: \n" + response)

        return response
    
    except Exception as e:
        print("========== EXCEPTION")
        print(traceback.format_exc())

        d = {
            "status": "error",
            "message": str(e),
            "traceback": traceback.format_exc()
        }

        print(d)

        return d
    

# 2. Setup Ngrok (Hardcoded for testing - replace with your actual token)
# Make sure to keep the string inside the quotes
ngrok.set_auth_token("3GibgmeF6xYrtfuJtyErnp4YrSY_4XvMFa5tE91HvYqG8V7yR")

# Close any existing tunnels to avoid "tunnel already exists" errors
ngrok.kill()

# 3. Connect and serve
public_url = ngrok.connect(8000)
print(f" * Public URL: {public_url.public_url}")

nest_asyncio.apply()
config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)
await server.serve()



 * Public URL: https://baguette-dismount-diocese.ngrok-free.dev                                     


INFO:     Started server process [5259]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


XXXXXXXX LOADING MODEL: Qwen/Qwen3.5-9B
Loading tokenizer...


config.json:   0%|          | 0.00/3.13k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


cuda:0
GPU memory allocated: 7.12 GB
GPU memory reserved: 7.22 GB
INFER BEGIN AT: 2026-07-20 17:33:25.774927


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


INFER END AT: 2026-07-20 17:34:09.422870
INFER TIME TAKEN: 0:00:43.648039
Peak GPU memory: 20.96 GB
======== INFERENCE: Sending response
INFO:     2401:4900:882d:ba13:23fb:833c:ec4f:e84:0 - "POST /plan HTTP/1.1" 200 OK
cuda:0
GPU memory allocated: 7.13 GB
GPU memory reserved: 30.85 GB
INFER BEGIN AT: 2026-07-20 17:34:57.401429
INFER END AT: 2026-07-20 17:35:39.122083
INFER TIME TAKEN: 0:00:41.720746
Peak GPU memory: 20.96 GB
======== INFERENCE: Sending response
INFO:     2401:4900:882d:ba13:23fb:833c:ec4f:e84:0 - "POST /plan HTTP/1.1" 200 OK
cuda:0
GPU memory allocated: 7.13 GB
GPU memory reserved: 30.85 GB
INFER BEGIN AT: 2026-07-20 17:37:12.244431
INFER END AT: 2026-07-20 17:38:09.777610
INFER TIME TAKEN: 0:00:57.533273
Peak GPU memory: 20.96 GB
======== INFERENCE: Sending response
INFO:     2401:4900:882d:ba13:23fb:833c:ec4f:e84:0 - "POST /plan HTTP/1.1" 200 OK
cuda:0
GPU memory allocated: 7.13 GB
GPU memory reserved: 30.85 GB
INFER BEGIN AT: 2026-07-20 18:00:08.705110
INFER END AT: